Quiz 46 — Hotel Occupancy Analysis  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(90)
room_types  = ["Standard","Superior","Deluxe","Suite","Penthouse"]
rates       = np.array([80, 120, 180, 350, 600])
capacity    = np.array([10, 8, 6, 4, 2])
occupancy   = np.random.randint(0, 10, (30, 5))

# ── 1. Nightly revenue ────────────────────────────────────────────────────────
# occupancy (30,5) × rates (5,) → broadcast last axis → (30,5)
revenue = occupancy * rates
print(f"Total 30-night revenue: £{revenue.sum():,.0f}")

# ── 2. Revenue per room type ─────────────────────────────────────────────────
rev_per_type = revenue.sum(axis=0)
for name, rev in zip(room_types, rev_per_type):
    print(f"  {name:12s}: £{rev:,.0f}")

# ── 3. Mean occupancy per night; 5 worst nights ──────────────────────────────
nightly_mean_occ  = occupancy.mean(axis=1)
worst5            = np.argsort(nightly_mean_occ)[:5]
print("\n5 worst occupancy nights:", worst5 + 1, "→ avg occ:", nightly_mean_occ[worst5].round(2))

# ── 4. Occupancy rate + low nights ───────────────────────────────────────────
# occupancy (30,5) / capacity (5,) → broadcast → (30,5)
occ_rate      = occupancy / capacity           # proportion per room type per night
avg_occ_rate  = occ_rate.mean(axis=1)          # mean across room types per night
low_nights    = np.where(avg_occ_rate < 0.30)[0]
print(f"\nLow nights (<30% avg occ): {len(low_nights)} nights")

# ── 5. Per room-type stats ────────────────────────────────────────────────────
print("\nPer room-type stats:")
for i, name in enumerate(room_types):
    col = occupancy[:, i]
    print(f"  {name:12s}: median={np.median(col):.1f}  "
          f"std={col.std():.2f}  peak_night={np.argmax(col)+1}")

# ── 6. Stacked bar chart by week ─────────────────────────────────────────────
# Pad to 28 nights (4 weeks of 7) then reshape
padded  = np.vstack([revenue, np.zeros((28 - 30 % 7 if 30%7 else 0, 5))])[:28]
weekly  = padded.reshape(4, 7, 5).sum(axis=1)       # (4,5)

x       = np.arange(4)
width   = 0.5
bottom  = np.zeros(4)
colours = ["steelblue","orange","green","tomato","purple"]

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, col) in enumerate(zip(room_types, weekly.T)):
    ax.bar(x, col, width, bottom=bottom, label=name, color=colours[i])
    bottom += col

ax.set_xticks(x)
ax.set_xticklabels([f"Week {i+1}" for i in range(4)])
ax.set_ylabel("Revenue (£)")
ax.set_title("Hotel Revenue by Room Type — Weekly (Stacked)")
ax.legend()
plt.tight_layout()
plt.savefig("hard_06_hotel_stacked.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# Occupancy rate = occupancy / capacity broadcasts because shapes align from right.
# Reshaping revenue into (4,7,5) then sum(axis=1) aggregates 7 days per week cleanly.